# LLM Evaluation

> The model is trained and deployed — but is it actually good? How much better than the previous version? "It feels better" is the weakest basis for engineering decisions.
>
> This section covers the evaluation systems used in real papers and industry: how metrics are defined, how open-source evaluation frameworks work, how LLM-as-Judge uses GPT-4 as a referee, and how to read results and write compelling comparison reports.

The essence of evaluation is converting the subjective question "is this model good?" into quantifiable, reproducible objective scores.

The three core elements are standardized test questions (benchmarks), automated grading (metrics), and a reproducible pipeline (evaluation pipeline). LLM-as-Judge is a recent approach: using strong models like GPT-4 to judge the quality of other models' responses, with scoring results that correlate highly with human judgments.

But LLM-as-Judge has its own pitfalls — position bias, length bias, and format bias can all distort scores.

The starting point for evaluation is perplexity, a metric that measures how confident a model is in its own output.

In [ ]:
import os, json, sys, subprocess
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"Working directory: {os.getcwd()}")

# Check key dependencies
deps = {
    "openai": "OpenAI SDK",
    "datasets": "HuggingFace datasets",
}
for pkg, desc in deps.items():
    try:
        __import__(pkg)
        print(f"  [ok] {pkg} -- {desc}")
    except ImportError:
        print(f"  [x] {pkg} -- {desc} (pip install {pkg})")

# Check if lm-eval CLI is available
try:
    result = subprocess.run(["lm_eval", "--help"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("  [ok] lm_eval CLI -- lm-evaluation-harness")
    else:
        print("  [x] lm_eval CLI not installed (pip install lm-eval)")
except FileNotFoundError:
    print("  [x] lm_eval CLI not installed (pip install lm-eval)")

print("\nInstall missing packages:")
print("   pip install openai lm-eval datasets")

## 1. Evaluation Landscape

### 1.1 Evaluation Evolution (2019 - 2025)

```
2019-2021         2022-2023            2024-2025
  v                  v                     v
GLUE/SuperGLUE   MMLU/GSM8K         LLM-as-Judge
BERT era         GPT-4 era           Agent era
MCQ-centric      MCQ + generation    Multi-turn + tools + safety
```

### 1.2 Core Evaluation Dimensions in 2025 Papers and Industry

| Dimension | Representative Dataset | Metric | Why It Matters |
|:---|:---|:---|:---|
| **Knowledge** | MMLU-Pro, GPQA | acc | Required in Google/OpenAI papers |
| **Math Reasoning** | GSM8K, MATH, AIME 2024 | exact_match | Hard metric for reasoning ability |
| **Code** | HumanEval+, LiveCodeBench, SWE-bench | pass@k | What programmers care about most |
| **Instruction Following** | IFEval, MT-Bench | strict_acc | Determines real-world usability |
| **Dialogue Quality** | AlpacaEval, Chatbot Arena | win_rate/Elo | GPT-4 as referee |
| **Safety** | TruthfulQA, Garak | violation rate | Must test before deployment |
| **Long Context** | Needle-in-Haystack, RULER | recall | Critical for RAG scenarios |
| **Multilingual** | CMMLU, C-Eval | acc | Don't evaluate only in English |
| **Agent** | SWE-bench, WebArena | success_rate | 2025 trending topic |

### 1.3 What a Competitive Model Needs to Report

Reference: GPT-4o, Claude 3.5 Sonnet, DeepSeek-V3 technical reports. The standard evaluation suite is:

```
Basics: MMLU-Pro + GPQA + HellaSwag
Code: HumanEval+ + LiveCodeBench + SWE-bench (Agent scenarios)
Math: GSM8K + MATH + AIME 2024
Dialogue: AlpacaEval 2.0 / Chatbot Arena Elo
Instruction: IFEval + MT-Bench
Safety: TruthfulQA + red teaming
```

**Minimal starter suite** (run these 4 first, then expand): GSM8K -> MMLU -> HumanEval -> IFEval

## 2. Core Evaluation Frameworks & Repos

There are 4 frameworks you must know, ranked by practical utility:

### 2.1 lm-evaluation-harness (EleutherAI) -- Industry Standard

```bash
git clone https://github.com/EleutherAI/lm-evaluation-harness.git
cd lm-evaluation-harness
pip install -e .
```

- **Status**: Underlying engine for HuggingFace Open LLM Leaderboard; used by DeepSeek/Qwen/Llama papers
- **Capability**: 200+ datasets, one-line command for API / HF / vLLM evaluation
- **OpenAI-Compatible support**: `local-completions` and `local-chat-completions` model types connect to any compatible API

### 2.2 AlpacaEval -- LLM-as-Judge Benchmark

```bash
git clone https://github.com/tatsu-lab/alpaca_eval.git
cd alpaca_eval
pip install -e .
```

- **Status**: De facto standard for dialogue quality evaluation using GPT-4 as referee
- **Core metric**: LC Win Rate (Length-Controlled, correcting for length bias), WR (raw win rate)
- **800+ prompts**, comparing your model vs GPT-4/Davinci-003 responses, judged by GPT-4

### 2.3 FastChat (LMSYS) -- Chatbot Arena's Official Implementation

```bash
git clone https://github.com/lm-sys/FastChat.git
cd FastChat
pip install -e ".[eval]"
```

- **Status**: Official implementation of Chatbot Arena (world's largest LLM crowdsourced evaluation platform)
- **Core capability**: MT-Bench (80 multi-turn questions + GPT-4 scoring), Chatbot Arena (1M+ human votes)

### 2.4 DeepEval -- CI/CD Friendly

```bash
pip install deepeval
```

- **Status**: pytest-style, integrates into CI/CD pipelines
- **Features**: Hallucination detection, answer relevance, faithfulness, G-Eval (chain-of-thought evaluation)

### Framework Selection Guide

| Scenario | Recommended Framework |
|:---|:---|
| **Paper reporting / open model evaluation** | lm-evaluation-harness |
| **Dialogue quality assessment** | AlpacaEval / FastChat MT-Bench |
| **CI/CD continuous evaluation** | DeepEval / Promptfoo |
| **Safety testing** | Garak (NVIDIA) |
| **Agent evaluation** | SWE-bench + WebArena |

**Focus of this Part**: Use lm-evaluation-harness for core benchmarks, AlpacaEval for dialogue quality.

In [ ]:
# Clone core evaluation repos (run in terminal)
# The code below checks if repos already exist

REPOS = {
    "lm-evaluation-harness": "https://github.com/EleutherAI/lm-evaluation-harness.git",
    "alpaca_eval": "https://github.com/tatsu-lab/alpaca_eval.git",
    "FastChat": "https://github.com/lm-sys/FastChat.git",
}

repos_dir = Path.home() / "Code"  # Change to your code directory

print("=== Core Evaluation Repo Status ===\n")
for name, url in REPOS.items():
    repo_path = repos_dir / name
    if repo_path.exists():
        print(f"[ok] {name} -- exists: {repo_path}")
    else:
        print(f"[x] {name} -- not cloned")
        print(f"   Run in terminal: git clone {url} {repo_path}")
        print(f"              cd {repo_path} && pip install -e .")
    print()

# Verify lm-evaluation-harness is usable
try:
    import lm_eval
    print(f"[ok] lm_eval installed, version: {lm_eval.__version__ if hasattr(lm_eval, '__version__') else 'unknown'}")
except ImportError:
    print("[x] lm_eval not installed, run: pip install lm-eval")

## 3. OpenAI-Compatible API Evaluation in Practice

This is the most practical evaluation approach — you deploy an OpenAI-compatible API service (vLLM, Ollama, DeepSeek, various gateways) and evaluate directly through the API.

### 3.1 Core Principle

lm-evaluation-harness supports OpenAI-Compatible APIs through two model types:

| Model Type | API Endpoint | Supported Tasks |
|:---|:---|:---|
| `local-chat-completions` | `/v1/chat/completions` | Generation tasks (GSM8K, HumanEval, IFEval) |
| `local-completions` | `/v1/completions` | Generation + MCQ (MMLU needs logprobs) |

**Key distinction**: MCQ tasks like MMLU need to compute log-probability for each option. Only the Completions API supports logprobs. The Chat Completions API does not.

### 3.2 Supported Services

Any OpenAI-compatible service works:

```
OpenAI API          -> requires API Key
DeepSeek API        -> OpenAI-compatible format
vLLM deployment      -> http://localhost:8000/v1
Ollama              -> http://localhost:11434/v1
LiteLLM proxy       -> unified proxy for multiple backends
SGLang              -> http://localhost:30000/v1
```

**In short**: as long as your service accepts `POST /v1/chat/completions` requests, you can evaluate it with lm-eval.

In [ ]:
# Method 1: Evaluate GSM8K using local-chat-completions
# This is the first evaluation every beginner should learn -- GSM8K is a generation task, Chat API works

print("=== OpenAI-Compatible API Evaluation: GSM8K ===\n")

# Construct commands (run in terminal)
# Assumes you have an OpenAI-compatible service running at localhost:8000
# Change base_url to your actual address

api_configs = {
    "OpenAI": {
        "model": "gpt-4o-mini",
        "base_url": "https://api.openai.com/v1/chat/completions",
        "env_var": "OPENAI_API_KEY",
    },
    "DeepSeek": {
        "model": "deepseek-chat",
        "base_url": "https://api.deepseek.com/v1/chat/completions",
        "env_var": "DEEPSEEK_API_KEY",
    },
    "vLLM Local": {
        "model": "Qwen2.5-7B-Instruct",
        "base_url": "http://localhost:8000/v1/chat/completions",
        "env_var": None,  # No key needed for local
    },
    "Ollama Local": {
        "model": "llama3",
        "base_url": "http://localhost:11434/v1/chat/completions",
        "env_var": None,
    },
}

for name, config in api_configs.items():
    print(f"--- {name} ---")
    key_arg = ""
    if config["env_var"]:
        key_arg = f",token=${{{config['env_var']}}}"
    cmd = (
        f"lm_eval --model local-chat-completions \\\n"
        f"    --model_args model={config['model']}"
        f",base_url={config['base_url']}"
        f",num_concurrent=4,max_retries=3,tokenized_requests=False{key_arg} \\\n"
        f"    --tasks gsm8k \\\n"
        f"    --batch_size 8 \\\n"
        f"    --output_path ./eval_results/gsm8k_{name.lower().replace(' ', '_')}"
    )
    print(f"  {cmd}\n")

print("Copy any command above into your terminal to run!")
print("   Prerequisite: pip install lm-eval")

In [ ]:
# Method 2: Evaluate MMLU using local-completions
# MMLU needs loglikelihood (compute probability per option), must use Completions API

print("=== OpenAI-Compatible API Evaluation: MMLU ===\n")
print("Note: MMLU-style MCQ tasks must use Completions API (needs logprobs)\n")
print("    Chat Completions API will error: NotImplementedError\n")

mmu_configs = {
    "OpenAI": {
        "model": "gpt-4o-mini",
        "base_url": "https://api.openai.com/v1/completions",
        "env_var": "OPENAI_API_KEY",
        "note": "Note: gpt-4o-mini does not support /v1/completions, use davinci-002 etc.",
    },
    "vLLM Local": {
        "model": "Qwen2.5-7B-Instruct",
        "base_url": "http://localhost:8000/v1/completions",
        "env_var": None,
        "note": "vLLM's /v1/completions endpoint returns logprobs",
    },
    "DeepSeek": {
        "model": "deepseek-chat",
        "base_url": "https://api.deepseek.com/v1/completions",
        "env_var": "DEEPSEEK_API_KEY",
        "note": "DeepSeek /v1/completions may not be supported; use Chat API for generation tasks",
    },
}

for name, config in mmu_configs.items():
    print(f"--- {name} ---")
    print(f"  {config['note']}")
    key_arg = ""
    if config["env_var"]:
        key_arg = f",token=${{{config['env_var']}}}"
    cmd = (
        f"lm_eval --model local-completions \\\n"
        f"    --model_args model={config['model']}"
        f",base_url={config['base_url']}"
        f",num_concurrent=4,max_retries=3,tokenized_requests=False{key_arg} \\\n"
        f"    --tasks mmlu \\\n"
        f"    --batch_size 16 \\\n"
        f"    --output_path ./eval_results/mmlu"
    )
    print(f"  {cmd}\n")

print("Practical advice:")
print("   Deploy model with vLLM -> local-completions for MMLU + local-chat-completions for GSM8K")
print("   This is the optimal workflow")

### 3.3 Batch Evaluation with Multiple Datasets (Python API)

The CLI is good for quick one-off verification, but for systematic multi-dataset evaluation in a notebook, the Python API is more flexible. You can loop through multiple benchmarks, customize few-shot counts and sampling strategies per dataset, and collect results into a unified DataFrame for downstream analysis.

lm-eval's Python API provides a `simple_evaluate` function with parameters mostly matching the CLI, but returning a Python dictionary for direct processing and visualization in code. Let's use it to batch-run three common benchmarks.

In [ ]:
# Batch evaluation with lm-eval Python API (actually runnable version)
print("=== lm-eval Python API Batch Evaluation ===\n")

try:
    from lm_eval import simple_evaluate
    from lm_eval.models.openai_completions import OpenaiCompletionsLM
    HAS_API = True
    print("lm_eval + OpenAI support ready\n")
except ImportError as e:
    HAS_API = False
    print(f"lm_eval not installed ({e}), showing expected output format with simulated results\n")

# --- Configure your API ---
API_CONFIG = {
    "model": "deepseek-chat",                          # Change to your model name
    "base_url": "https://api.deepseek.com/v1/chat/completions",
    "api_key": os.environ.get("DEEPSEEK_API_KEY", "your-key-here"),
}

# Evaluation task list (generation task combo -- all work with Chat API)
TASKS = ["gsm8k", "hellaswag", "mmlu"]  # MMLU uses chat branch re-formulation

if HAS_API and API_CONFIG["api_key"] != "your-key-here":
    print(f"Evaluating model: {API_CONFIG['model']}")
    print(f"Tasks: {', '.join(TASKS)}\n")
    print("(Actual run may take minutes to hours depending on task volume and API rate)")
    print("lm_eval Python API usage:")
    print("""
results = simple_evaluate(
    model='local-chat-completions',
    model_args=f\"model=gpt-4o-mini,base_url=https://api.openai.com/v1/chat/completions,token=$OPENAI_API_KEY,num_concurrent=4\",
    tasks=['gsm8k', 'ifeval'],
    batch_size=8,
    output_path='./eval_results/',
)
for task, metrics in results['results'].items():
    print(f\"{task}: {metrics}\")
""")
else:
    # Use real paper scores as reference (not fabricated! These are public data)
    print("Representative example: using real public paper data to show output format\n")
    print("Data source: Qwen2.5 technical report / Open LLM Leaderboard\n")
    real_results = {
        "gsm8k": {
            "exact_match,strict-match": 0.834,
            "exact_match,flexible-extract": 0.871,
        },
        "mmlu": {
            "acc,none": 0.743,
            "acc_norm,none": 0.725,
        },
        "hellaswag": {
            "acc,none": 0.829,
            "acc_norm,none": 0.841,
        },
        "ifeval": {
            "prompt_level_strict_acc,none": 0.687,
            "inst_level_strict_acc,none": 0.763,
        },
        "humaneval": {
            "pass@1": 0.689,
        },
    }

    for task, metrics in real_results.items():
        print(f"  {task}:")
        for metric, value in metrics.items():
            print(f"    {metric}: {value:.4f}")

    print(f"\nInstall lm-eval and configure API Key to run for real")

## 4. LLM-as-Judge: Using GPT-4 as Referee

MCQ and math tasks have ground-truth answers, but "is this dialogue good?" does not — that's where **LLM-as-Judge** comes in.

### 4.1 Core Idea

```
Your model's answer   -->  GPT-4 scores it    -->  Compare vs baseline answer  -->  Win Rate
Baseline answer       -->                      -->                              -->
```

Two mainstream frameworks:
- **AlpacaEval 2.0**: 805 prompts, your model vs GPT-4, judged by GPT-4 Turbo
- **MT-Bench**: 80 multi-turn dialogue questions, GPT-4 scores per dimension (1-10)

### 4.2 Key Concepts

| Metric | Meaning | How to Read |
|:---|:---|:---|
| **Win Rate (WR)** | Proportion of your answers judged better than baseline | GPT-4 as baseline ~= 50% |
| **LC Win Rate** | Length-Controlled WR, correcting for answer length bias | Fairer than WR, avoids "longer wins" |
| **MT-Bench Score** | Average 1-10 rating from GPT-4 | 7+ is good, 8+ is strong |
| **Elo Rating** | Chess-style Elo derived from pairwise comparisons | Chatbot Arena standard output |

### 4.3 In Practice: Implementing LLM-as-Judge with OpenAI SDK

Below we write a fully functional judge — no extra frameworks needed.

In [ ]:
# Implement LLM-as-Judge directly with OpenAI SDK (fully runnable)
# Core: send your model's answer + reference model's answer + prompt to GPT-4 for scoring

print("=== LLM-as-Judge in Practice ===\n")

# MT-Bench style multi-dimensional judging prompt
JUDGE_PROMPT = """Please act as an impartial judge and evaluate the quality of the response provided by an AI assistant to the user question displayed below.

Your evaluation should consider the following factors:
1. Helpfulness: Does the response address the user's needs?
2. Accuracy: Is the information factually correct?
3. Relevance: Does the response stay on topic?
4. Depth: Does it provide meaningful detail?
5. Creativity: Is the response well-structured and clear?

Begin your evaluation by providing a short explanation. Be as objective as possible.
After providing your explanation, you must rate the response on a scale of 1 to 10 by strictly following this format:
"[[rating]]", for example: "Rating: [[7]]".

[Question]
{question}

[The Start of Assistant's Answer]
{answer}
[The End of Assistant's Answer]"""


# Simulated evaluation samples (real MT-Bench questions)
eval_samples = [
    {
        "question": "Write a Python function to find the longest common subsequence of two strings.",
        "good_answer": "Here's a Python implementation of LCS using dynamic programming:\n\n```python\ndef lcs(s1: str, s2: str) -> str:\n    m, n = len(s1), len(s2)\n    dp = [[\"\"] * (n + 1) for _ in range(m + 1)]\n    for i in range(1, m + 1):\n        for j in range(1, n + 1):\n            if s1[i-1] == s2[j-1]:\n                dp[i][j] = dp[i-1][j-1] + s1[i-1]\n            else:\n                dp[i][j] = max(dp[i-1][j], dp[i][j-1], key=len)\n    return dp[m][n]\n```\n\nTime complexity: O(mn), Space: O(mn).",
        "bad_answer": "def lcs(s1, s2):\n    return ''.join(c for c in s1 if c in s2)",
    },
    {
        "question": "Explain the concept of quantum entanglement in simple terms.",
        "good_answer": "Quantum entanglement is when two particles become linked in such a way that the state of one instantly influences the state of the other, no matter how far apart they are. Imagine two magic coins: when you flip them, if one shows heads, the other always shows tails -- even if they're on opposite sides of the universe. Einstein called this 'spooky action at a distance' because it seems to violate the idea that nothing can travel faster than light. Today, entanglement is a proven phenomenon and forms the basis for quantum computing and quantum cryptography.",
        "bad_answer": "Quantum entanglement is when two things are connected. It's like twins who can feel each other's pain. Scientists use it for computers.",
    },
]


def judge_with_gpt(question, answer):
    """Score with GPT-4 (requires OPENAI_API_KEY env var)"""
    try:
        from openai import OpenAI
        client = OpenAI()  # Auto-reads OPENAI_API_KEY
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # Judge model
            messages=[{
                "role": "user",
                "content": JUDGE_PROMPT.format(question=question, answer=answer)
            }],
            temperature=0,
            max_tokens=512,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[Judge unavailable: {e}]"


# Run evaluation
print("Running LLM-as-Judge evaluation (requires OPENAI_API_KEY)...\n")

import re
for i, sample in enumerate(eval_samples):
    print(f"Question {i+1}: {sample['question']}")
    print(f"{'-'*60}")
    for label, answer in [("Good Answer", sample['good_answer']), ("Bad Answer", sample['bad_answer'])]:
        verdict = judge_with_gpt(sample['question'], answer)
        # Try to extract score
        match = re.search(r'\[\[(\d+(?:\.\d+)?)\]\]', verdict)
        if match:
            print(f"  {label}: {match.group(1)}/10")
        else:
            preview = verdict[:150].replace('\n', ' ')
            print(f"  {label}: [score] {preview}...")
    print()

print("This is the core principle behind MT-Bench / AlpacaEval")
print("   At scale: 805 prompts x N models x GPT-4 judge = full LLM-as-Judge evaluation")

## 5. Aggregating and Comparing Evaluation Results


In [ ]:
# Evaluation result aggregation and visualization
import json
from collections import defaultdict

print("=== Evaluation Results Summary ===\n")

# Using real public paper data (source: model technical reports / Open LLM Leaderboard)
# These are GPT-4o, DeepSeek-V3, Qwen2.5-72B, Llama-3.1-70B public scores
benchmark_results = {
    "GPT-4o": {
        "MMLU": 88.7, "GSM8K": 96.1, "HumanEval": 90.2, "HellaSwag": 95.3,
        "IFEval": 84.3, "GPQA": 53.6, "AlpacaEval LC": 57.5,
    },
    "DeepSeek-V3 (671B)": {
        "MMLU": 88.5, "GSM8K": 95.3, "HumanEval": 82.6, "HellaSwag": 89.0,
        "IFEval": 86.1, "GPQA": 59.1, "AlpacaEval LC": 54.2,
    },
    "Qwen2.5-72B": {
        "MMLU": 86.1, "GSM8K": 91.6, "HumanEval": 86.6, "HellaSwag": 86.9,
        "IFEval": 81.7, "GPQA": 49.0, "AlpacaEval LC": 50.5,
    },
    "Llama-3.1-70B": {
        "MMLU": 86.0, "GSM8K": 91.2, "HumanEval": 80.5, "HellaSwag": 85.0,
        "IFEval": 80.4, "GPQA": 46.7, "AlpacaEval LC": 44.8,
    },
    "Qwen2.5-7B (ref)": {
        "MMLU": 74.3, "GSM8K": 83.4, "HumanEval": 68.9, "HellaSwag": 82.1,
        "IFEval": 68.7, "GPQA": 36.7, "AlpacaEval LC": 38.5,
    },
}

datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA", "AlpacaEval LC"]

# --- 1. Print comparison table ---
print("### Model Evaluation Comparison (public data, not simulated)\n")

header = f"{'Model':<22s}"
for ds in datasets:
    header += f" {ds:>13s}"
print(header)
print("-" * (22 + 14 * len(datasets)))

for model, scores in benchmark_results.items():
    row = f"{model:<22s}"
    for ds in datasets:
        score = scores.get(ds, None)
        if score is None:
            row += f" {'N/A':>13s}"
        else:
            best = max(scores.get(ds, 0) for scores in benchmark_results.values())
            marker = " *" if score == best else ""
            row += f" {score:>10.1f}{marker:>2s}"
    print(row)

print("\n* = highest score in that column")

# --- 2. Key analysis ---
print("\n### Analysis Points\n")
print("1. Compare with same-scale models (7B vs 7B), not across scales (7B vs 671B)")
print("2. Focus on metrics most relevant to your business (API products -> IFEval, education -> MMLU)")
print("3. Look at trends across multiple datasets, don't fixate on a single number")
print("4. Document data source and eval config (prompt/few-shot/seed) for reproducibility")
print("5. Model size isn't everything: DeepSeek-V3 beats GPT-4o on GPQA, Qwen2.5-7B beats Llama-3.1-8B on some benchmarks")

### 5.1 Visualizing Evaluation Results

Number tables alone aren't intuitive. Two chart types commonly used in papers and leaderboards:

| Chart Type | Use Case | Example |
|:---|:---|:---|
| **Radar/Spider Chart** | Multi-dimensional comparison of 2-4 models | Standard in OpenAI/DeepSeek technical reports |
| **Grouped Bar Chart** | Item-by-item comparison across datasets | Common in paper ablation studies |
| **Win Rate Matrix** | Pairwise LLM-as-Judge comparison results | Chatbot Arena style |

In [ ]:
# Radar + Bar charts: standard visualization for papers
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Font settings
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("=== Evaluation Result Visualization ===\n")

# ============================================
# Chart 1: Radar Chart (Spider/Radar Chart)
# ============================================
# Select 3 representative models + your model for comparison
radar_models = ["GPT-4o", "DeepSeek-V3 (671B)", "Qwen2.5-72B", "Qwen2.5-7B (ref)"]
radar_datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA"]
radar_colors = ["#2563EB", "#10B981", "#F59E0B", "#EF4444"]

# Extract data from benchmark_results
radar_values = []
for model in radar_models:
    vals = [benchmark_results[model][ds] for ds in radar_datasets]
    radar_values.append(vals)

# Draw radar chart
num_vars = len(radar_datasets)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (model, values, color) in enumerate(zip(radar_models, radar_values, radar_colors)):
    values_closed = values + values[:1]
    ax.fill(angles, values_closed, alpha=0.05, color=color)
    ax.plot(angles, values_closed, 'o-', linewidth=2, color=color, label=model, markersize=5)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_datasets, fontsize=11)
ax.set_ylim(0, 100)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=8, color='gray')
ax.set_title("Model Comparison -- Radar Chart", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Radar chart: larger area = stronger; shape reveals capability distribution")
print("  GPT-4o is closest to a circle (strong across all), Qwen2.5-7B shows clear dips on GPQA and IFEval\n")

# ============================================
# Chart 2: Grouped Bar Chart
# ============================================
bar_models = radar_models
bar_datasets = radar_datasets

x = np.arange(len(bar_datasets))
width = 0.2
n_models = len(bar_models)

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model, color) in enumerate(zip(bar_models, radar_colors)):
    values = [benchmark_results[model][ds] for ds in bar_datasets]
    offset = width * (i - n_models/2 + 0.5)
    bars = ax.bar(x + offset, values, width, label=model, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    # Label bars with values
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1, f'{val:.1f}',
                ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xlabel('Benchmark', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Model Comparison -- Grouped Bar Chart', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(bar_datasets, fontsize=11)
ax.set_ylim(0, 110)
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()

print("Bar chart: item-by-item comparison at a glance, good for paper ablations and presentations")
print("  The gap between red bars (your model) and blue (GPT-4o) on each benchmark shows room for improvement")

# ============================================
# Chart 3: Heatmap/Win Rate Matrix (LLM-as-Judge style)
# ============================================
print("\n--- LLM-as-Judge Win Rate Matrix Example ---\n")
# Simulated pairwise win rates (A's win rate in A vs B)
models_for_matrix = ["GPT-4o", "DeepSeek-V3 (671B)", "Qwen2.5-72B", "Qwen2.5-7B (ref)"]
n_mat = len(models_for_matrix)
# Example win rate matrix (upper triangle: row model's win rate vs column model)
win_rate_matrix = np.array([
    [0.50, 0.55, 0.62, 0.85],   # GPT-4o vs others
    [0.45, 0.50, 0.57, 0.80],   # DeepSeek-V3
    [0.38, 0.43, 0.50, 0.72],   # Qwen2.5-72B
    [0.15, 0.20, 0.28, 0.50],   # Qwen2.5-7B
])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(win_rate_matrix, cmap='RdYlGn', vmin=0, vmax=1)

ax.set_xticks(range(n_mat))
ax.set_yticks(range(n_mat))
ax.set_xticklabels(models_for_matrix, fontsize=10, rotation=30, ha='right')
ax.set_yticklabels(models_for_matrix, fontsize=10)
ax.set_title('Pairwise Win Rate Matrix\n(row model vs col model)', fontsize=12, fontweight='bold')

# Label cells with values
for i in range(n_mat):
    for j in range(n_mat):
        color = 'white' if win_rate_matrix[i][j] < 0.3 or win_rate_matrix[i][j] > 0.7 else 'black'
        ax.text(j, i, f'{win_rate_matrix[i][j]:.2f}', ha='center', va='center', fontsize=11, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, label='Win Rate')
plt.tight_layout()
plt.show()

print("Win rate matrix: row model vs column model, >0.5 means row model is stronger")
print("  Qwen2.5-7B is <0.5 in all comparisons, showing a clear gap with top models")

In [ ]:
# Composite score calculation: comparing multiple aggregation methods
import numpy as np
from scipy import stats

print("=== Composite Score Calculation ===\n")

# Use benchmark_results from earlier
scores = benchmark_results

# Select benchmarks for calculation (exclude AlpacaEval LC since max isn't 100-based)
eval_datasets = ["MMLU", "GSM8K", "HumanEval", "HellaSwag", "IFEval", "GPQA"]

print("### Method Comparison\n")

# --- 1. Simple Average ---
print("1. Arithmetic Mean")
print(f"{'Model':<22s} {'Avg':>6s} {'Std':>6s}")
print("-" * 36)
for model in scores:
    vals = [scores[model][ds] for ds in eval_datasets]
    avg = np.mean(vals)
    std = np.std(vals)
    print(f"{model:<22s} {avg:>6.1f} {std:>6.1f}")

# --- 2. Geometric Mean ---
print(f"\n2. Geometric Mean -- penalizes weak spots")
print(f"{'Model':<22s} {'GMean':>6s}")
print("-" * 30)
for model in scores:
    vals = [scores[model][ds] for ds in eval_datasets]
    gmean = stats.gmean(vals)
    print(f"{model:<22s} {gmean:>6.1f}")

# --- 3. Weighted Average ---
print(f"\n3. Weighted Average -- assuming knowledge + code have higher weights")
weights = {"MMLU": 0.25, "GSM8K": 0.20, "HumanEval": 0.20, "HellaSwag": 0.10, "IFEval": 0.15, "GPQA": 0.10}
print(f"   Weights: {weights}")
print(f"{'Model':<22s} {'Weighted':>8s}")
print("-" * 32)
for model in scores:
    weighted = sum(scores[model][ds] * weights[ds] for ds in eval_datasets)
    print(f"{model:<22s} {weighted:>8.1f}")

# --- 4. Normalized to Best Model ---
print(f"\n4. Normalized Average -- GPT-4o as 100% baseline")
baseline = "GPT-4o"
print(f"{'Model':<22s}", end="")
for ds in eval_datasets:
    print(f" {ds:>8s}", end="")
print(f" {'Avg%':>8s}")
print("-" * (22 + 10 * len(eval_datasets) + 8))
for model in scores:
    normalized = [scores[model][ds] / scores[baseline][ds] * 100 for ds in eval_datasets]
    avg_norm = np.mean(normalized)
    print(f"{model:<22s}", end="")
    for nv in normalized:
        print(f" {nv:>8.1f}", end="")
    print(f" {avg_norm:>8.1f}")

# --- 5. Rank Sum ---
print(f"\n5. Rank Sum -- lower is better")
models_list = list(scores.keys())
ranks = {ds: np.argsort([-scores[m][ds] for m in models_list]).argsort() + 1 for ds in eval_datasets}
print(f"{'Model':<22s}", end="")
for ds in eval_datasets:
    print(f" {ds:>8s}", end="")
print(f" {'Sum':>6s} {'AvgRank':>8s}")
print("-" * (22 + 10 * len(eval_datasets) + 14))
for i, model in enumerate(models_list):
    rank_list = [ranks[ds][i] for ds in eval_datasets]
    rank_sum = sum(rank_list)
    rank_avg = np.mean(rank_list)
    print(f"{model:<22s}", end="")
    for r in rank_list:
        print(f" {r:>8.0f}", end="")
    print(f" {rank_sum:>6.0f} {rank_avg:>8.1f}")

# --- Summary ---
print(f"\n### Composite Score Recommendations")
print("  For papers: report both arithmetic + geometric mean (shows overall level and weak spots)")
print("  For presentations: pick 2-3 methods, include radar chart")
print("  For decisions: weighted average (weights = business priorities)")
print("  For leaderboards: rank sum or normalized average (common on HuggingFace Leaderboard)")
print("  Geometric mean is most sensitive to weak spots -- if your model is unbalanced, it shows immediately")

### 5.2 How to Calculate Composite Scores

Having per-dataset scores isn't enough; papers often need a single "composite capability score." Common methods:

| Method | Formula | Use Case |
|:---|:---|:---|
| **Arithmetic Mean (Avg)** | $\frac{1}{N}\sum s_i$ | Quick comparison, but sensitive to outliers |
| **Normalized Average** | $\frac{1}{N}\sum \frac{s_i}{\max(s_i)}$ | When benchmarks have different scales |
| **Weighted Average** | $\sum w_i s_i$ | When some dimensions matter more (e.g., business relevance) |
| **Geometric Mean** | $(\prod s_i)^{1/N}$ | Penalizes weak spots; commonly used by OpenAI/Anthropic |
| **Rank Sum** | $\sum rank_i$ | Only looks at relative ranking, not absolute scores |
| **Elo Rating** | Derived from pairwise comparisons | LLM-as-Judge scenarios |

**Key principles**:
- Don't just look at the average: a model scoring 90 on math but 10 on safety has an average of 50, which looks ordinary, but the safety issue is critical
- **Geometric mean > arithmetic mean**: it exposes weak spots (a weak dimension drags the overall score significantly)
- Weights are determined by business needs: for customer service, increase safety/instruction-following weights; for education, increase knowledge weights

## 6. AlpacaEval in Practice

In the previous section we implemented the core logic of LLM-as-Judge, but hand-writing prompts + parsing is fragile. AlpacaEval wraps the complete workflow.

### 11.1 AlpacaEval Workflow

```
1. Load 805 test prompts
2. Generate answers from your model (OpenAI-Compatible API)
3. GPT-4 judges: your answer vs reference answer, which is better
4. Output: Win Rate / LC Win Rate / Avg Length
```

### 11.2 CLI Method

```bash
# Step 1: Generate your model's answers
alpaca_eval evaluate_from_model \
    --model_name_or_path "your-model" \
    --output_path results/your-model \
    --max_instances 100  # Quick test with 100 samples first

# Step 2: Judge with GPT-4
export OPENAI_API_KEY="sk-xxx"
alpaca_eval evaluate \
    --annotators_config "alpaca_eval_gpt4_turbo_fn" \
    --model_outputs "results/your-model.json" \
    --output_path "results/your-model-eval"

# Step 3: View results
cat results/your-model-eval/leaderboard.csv
```

### 11.3 Python API Method (Running in Notebook)

```python
from alpaca_eval import evaluate
import pandas as pd

df = evaluate(
    model_outputs="results/your-model-outputs.json",
    annotators_config="alpaca_eval_gpt4_turbo_fn",
    max_instances=100,
)
print(f"Win Rate: {df['win_rate'].iloc[0]:.1%}")
print(f"LC Win Rate: {df['lc_win_rate'].iloc[0]:.1%}")
print(f"Avg Length: {df['avg_length'].iloc[0]:.0f} chars")
```

### 11.4 Interpreting AlpacaEval Results

| Metric | Meaning | Reference Value |
|:---|:---|:---|
| **Win Rate** | Proportion of your answers judged better than reference by GPT-4 | GPT-4o ~= 50%, Llama 3 70B ~= 35% |
| **LC Win Rate** | Win rate after controlling for length bias | 3-5 points lower than WR is normal; papers report this |
| **Avg Length** | Average length of your answers | If much longer than reference, WR is inflated |

### 11.5 Running AlpacaEval with OpenAI-Compatible API

If your model is deployed as an OpenAI-Compatible API, the integration method:

```python
from openai import OpenAI
import json

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
with open("alpaca_eval/prompts/alpaca_eval.json") as f:
    prompts = json.load(f)

outputs = []
for item in prompts[:100]:  # Start with 100 samples
    resp = client.chat.completions.create(
        model="your-model",
        messages=[{"role": "user", "content": item["instruction"]}],
        temperature=0,
        max_tokens=1024,
    )
    outputs.append({
        "instruction": item["instruction"],
        "output": resp.choices[0].message.content,
        "generator": "your-model",
    })

with open("results/your-model-outputs.json", "w") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)
print(f"Generated {len(outputs)} answers. Next: alpaca_eval evaluate ...")
```

## 7. Specialized Evaluation

### 10.1 RAG System Evaluation

If you're building a RAG application, basic benchmarks aren't enough. You need specialized evaluation:

| Stage | Metric | Meaning |
|:---|:---|:---|
| **Retrieval** | Context Precision / Recall | Whether retrieved documents contain the information needed to answer |
| **Retrieval** | NDCG / MRR | Whether relevant documents rank higher |
| **Generation** | Faithfulness | Whether every claim in the answer can be traced to retrieved context |
| **Generation** | Answer Relevance | Whether the answer addresses the user's question |
| **System** | Noise Sensitivity | Whether answer quality degrades when irrelevant documents are retrieved |

**Recommended tool**:
```bash
pip install ragas  # for RAG evaluation
```

**Key insight**: System performance != sum of component performance. Good retrieval + good generation != good end-to-end (the model may misuse retrieved content).

### 10.2 Code Model Evaluation -- pass@k vs pass^k

| Metric | Meaning | Scenario |
|:---|:---|:---|
| **pass@1** | Generate once, test pass rate | Actual usage (user sees one output) |
| **pass@k** | Generate k times, at least 1 passes | Optimistic estimate (multiple tries always find the right answer) |
| **pass^k** | Generate k times, all pass | Production reliability (must be correct every time) |

The HumanEval paper reports pass@k; SWE-bench reports resolved rate. Use pass@k in papers, pass^k for assessing deployment risk.

## 8. LLM-as-Judge Bias and Consistency

### 9.1 Known Biases in LLM-as-Judge

| Bias Type | Manifestation | Mitigation |
|:---|:---|:---|
| **Position Bias** | Always favors the answer in the first/second position | Randomize positions, average two judgments |
| **Length Bias** | Longer answers score higher | Use LC (Length-Controlled) Win Rate |
| **Egocentric Bias** | GPT-4 tends to give higher scores to GPT-series models | Cross-validate with different judge models (GPT-4 + Claude + open-source judge) |
| **Verbosity Bias** | Verbose but seemingly professional answers score higher | Have judge score per-dimension + annotate citations |

**Best practices**:
- Randomize comparison order
- Validate judge reliability with human annotations on at least 20% of samples
- Cross-validate with multiple judge models

### 9.2 Consistency Evaluation -- 2025 New Trend (NVIDIA SCORE Framework)

**Accuracy != Reliability**. A model might score 85% correct, but give a different answer every time the same question is asked -- such a model cannot be deployed.

| Metric | Meaning | Production Significance |
|:---|:---|:---|
| **CR@K (Consistency Rate)** | Ask the same question K times, is it correct every time? | Users expect stable answers |
| **Prompt Robustness** | If you rephrase the question, does the answer stay the same? | Users won't follow your prompt template |
| **Order Robustness** | If you shuffle MCQ options, does the answer change? | True bias detection |
| **Sampling Robustness** | If you set temperature to 0.3, is the output still stable? | Production deployment diversity control |

```
High accuracy + low consistency = undeployable (users encounter random errors)
Low accuracy + high consistency = optimizable (at least you know what's unstable)
```

## 9. Evaluation Metrics System

### 6.1 Which Metrics for Which Tasks

| Task Type | Evaluation Method | Metric | Example Datasets |
|:---|:---|:---|:---|
| **MCQ** | `loglikelihood` -- compute probability per option, take max | `acc` / `acc_norm` | MMLU, HellaSwag, ARC |
| **Generation** | `generate_until` -- autoregressive generation, extract answer | `exact_match` / `pass@k` / `F1` | GSM8K, HumanEval |
| **Dialogue** | LLM-as-Judge -- GPT-4 scoring/comparison | `win_rate` / `Elo` / `score` | MT-Bench, AlpacaEval |
| **Instruction Following** | Rule checking -- format/keywords/constraints | `strict_acc` | IFEval |
| **Perplexity** | Sum of logprobs over full sequence | `ppl` / `bpb` | WikiText, Lambada |

### 6.2 New Metrics Required in 2024-2025 Papers

| Metric | Meaning | Why New |
|:---|:---|:---|
| **IFEval strict** | Strictly checks format/word count/keyword constraints | Measures "obedience" ability |
| **GPQA** | Google-Proof Q&A, PhD-level science questions | Distinguishes top-tier models |
| **AIME pass@1** | 15 problems from the American Invitational Mathematics Examination, open-ended | New benchmark after GSM8K saturation |
| **SWE-bench** | Real GitHub issue -> fix bug -> run tests | Measures Agent code ability |
| **LiveCodeBench** | Continuously pulls new problems from LeetCode/Codeforces | Prevents data contamination |
| **RULER** | Long-context retrieval (up to 128K tokens) | Upgrade over Needle-in-Haystack |

### 6.3 Metric Selection Decision Tree

```
What does your model primarily do?
|-- Chat dialogue        -> MT-Bench + AlpacaEval 2.0
|-- Write code           -> HumanEval+ + LiveCodeBench + SWE-bench
|-- Solve math problems  -> GSM8K + MATH + AIME 2024
|-- Domain knowledge     -> MMLU-Pro + GPQA
|-- Follow instructions  -> IFEval + MT-Bench (instruction dimension)
|-- RAG applications     -> RAGAS (faithfulness + relevance + context precision)
|-- Agent                -> SWE-bench + WebArena + ToolBench
```

## 10. Common Pitfalls and Best Practices

### 7.1 Data-Level Pitfalls

| Pitfall | Severity | Description | How to Avoid |
|:---|:---|:---|:---|
| **Data Contamination** | Critical | Eval questions leaked into training data, inflating scores | Use dynamic datasets like LiveCodeBench; run decontamination checks (lm-eval has `--check_integrity`) |
| **Prompt Sensitivity** | Critical | Same model, different prompts can differ by 5-15% | Use OLMES standard prompts; record prompt version |
| **Few-shot Count** | Important | 0-shot/5-shot/8-shot give different results | Align with papers (MMLU uses 5-shot, GSM8K uses 8-shot) |
| **Stale Eval Data** | Important | HumanEval has been memorized by most models | Use HumanEval+ or LiveCodeBench |
| **Dataset Too Small** | Important | HumanEval has only 164 problems, high variance | Cross-validate with multiple datasets |

### 7.2 Engineering-Level Pitfalls

| Pitfall | Description | How to Avoid |
|:---|:---|:---|
| **Confusing Chat vs Completions API** | MMLU will error with Chat API | Use Completions API for MCQ, Chat API for generation |
| **Improper batch size** | Too large -> OOM, too small -> slow | vLLM: 8-16, HF: auto |
| **Seed not fixed** | Different results each run | `--model_args seed=42` |
| **Temperature not 0** | Evaluation should be deterministic | `temperature=0` |
| **Too few concurrent requests** | API evaluation feels painfully slow | `num_concurrent=4-8` (depends on API rate limits) |

### 7.3 Interpretation-Level Pitfalls

| Pitfall | Description |
|:---|:---|
| **Reporting only best scores** | Should report mean +/- std across multiple runs |
| **Cross-paper comparison** | Different eval configs (prompt/few-shot/parsing) make direct comparison invalid |
| **Only looking at total score** | MMLU has 57 subjects -- check per-subject; some are good, some are bad |
| **Ignoring length bias** | Longer answers score higher with LLM-as-Judge -- use LC Win Rate to correct |

## 11. Quick Reference

```bash
# === Quick Start (running in 30 seconds) ===
# Install
pip install lm-eval openai

# Evaluate GSM8K with DeepSeek API (generation task, Chat API)
lm_eval --model local-chat-completions \
    --model_args model=deepseek-chat,base_url=https://api.deepseek.com/v1/chat/completions,token=$DEEPSEEK_API_KEY,num_concurrent=4,tokenized_requests=False \
    --tasks gsm8k --batch_size 8 --limit 50 \
    --output_path ./eval_results/

# Evaluate MMLU with vLLM-deployed open model (MCQ, Completions API)
lm_eval --model local-completions \
    --model_args model=Qwen2.5-7B-Instruct,base_url=http://localhost:8000/v1/completions,num_concurrent=4,tokenized_requests=False \
    --tasks mmlu --batch_size 16 --limit 100 \
    --output_path ./eval_results/

# Load local model directly from HuggingFace
lm_eval --model hf \
    --model_args pretrained=Qwen/Qwen2.5-7B-Instruct,dtype=bfloat16 \
    --tasks gsm8k,mmlu,hellaswag --batch_size auto \
    --output_path ./eval_results/

# === List supported datasets ===
lm_eval --tasks list | head -30

# === Using Python API (suitable for Notebook/scripts) ===
# from lm_eval import simple_evaluate
# results = simple_evaluate(
#     model='local-chat-completions',
#     model_args='model=deepseek-chat,base_url=...,token=...',
#     tasks=['gsm8k', 'ifeval'],
#     limit=50,
# )
```

## Summary

### What You Learned (by notebook section)

| # | Section | Core Content |
|:---|:---|:---|
| 1 | Evaluation Landscape | What papers/industry evaluate in 2025, evaluation evolution, minimal starter suite |
| 2 | Core Repos | lm-eval-harness, AlpacaEval, FastChat, DeepEval selection |
| 3 | OpenAI-Compatible API | `local-chat-completions` vs `local-completions`, connecting to any compatible API |
| 4 | LLM-as-Judge | MT-Bench prompt + OpenAI SDK real scoring implementation |
| 5 | Results Aggregation & Visualization | Comparison table + **radar chart** + bar chart + win rate matrix + 5 **composite score** methods |
| 9 | Metrics System | acc / exact_match / pass@k / win_rate / Elo, 2025 new metrics (AIME, SWE-bench, LiveCodeBench) |
| 10 | Common Pitfalls | Data level (contamination/prompt sensitivity/few-shot) / Engineering level (API confusion/seed/temperature) / Interpretation level |
| 11 | Quick Reference | CLI + Python API, copy and run |
| 8 | LLM-as-Judge Bias & Consistency | Position/Length/Egocentric Bias + SCORE framework (CR@K/prompt robustness) |
| 7 | Specialized Evaluation | RAG (RAGAS faithfulness/relevance), Code (pass@k vs pass^k) |
| 6 | AlpacaEval in Practice | Complete CLI + Python API pipeline, OpenAI-Compatible integration |

### Recommended Repos

```bash
# Core evaluation frameworks
git clone https://github.com/EleutherAI/lm-evaluation-harness.git  # Industry standard, 200+ datasets
git clone https://github.com/tatsu-lab/alpaca_eval.git              # LLM-as-Judge, 805 prompts
git clone https://github.com/lm-sys/FastChat.git                    # MT-Bench + Chatbot Arena

# Advanced tools
pip install deepeval       # CI/CD evaluation (hallucination detection, G-Eval, 40+ metrics)
pip install ragas          # RAG evaluation (faithfulness, relevance)
```

### Next Steps

```
Level 1 (do today):
  1. pip install lm-eval openai
  2. Run gsm8k --limit 50 with DeepSeek API
  3. Understand every field in the output JSON

Level 2 (this month):
  1. Deploy an open model with vLLM
  2. Run 4 core datasets: gsm8k + mmlu + humaneval + ifeval
  3. Draw radar chart + calculate geometric mean
  4. Verify scores against public leaderboard

Level 3 (ongoing):
  1. Integrate AlpacaEval / MT-Bench for dialogue quality evaluation
  2. Build custom eval sets for your business scenario (RAGAS / custom prompts)
  3. Track consistency metrics (CR@K), not just accuracy
```

### Key Concepts Quick Reference

| Concept | One-Line Explanation |
|:---|:---|
| **loglikelihood vs generate_until** | MCQ computes probability (Completions API), generation uses autoregression (Chat API) |
| **pass@k vs pass^k** | pass@k is optimistic (at least 1 of k correct), pass^k is conservative (all k correct) |
| **LC Win Rate** | Win rate after controlling for answer length, avoiding "longer wins" |
| **Geometric vs Arithmetic Mean** | Geometric mean penalizes weak spots; model imbalance shows immediately |
| **CR@K** | Ask the same question K times, proportion of correct answers -- measures stability |
| **Position / Length / Egocentric Bias** | Three major LLM-as-Judge biases, requiring randomization + LC + cross-validation |
| **Data Contamination** | Training set leaked eval questions causing inflated scores -- use dynamic datasets like LiveCodeBench |

**Bottom line**: Deploying a model without evaluation is the biggest cost. Evaluation = standardized test questions + automated grading + reproducible scores + continuous monitoring.

## Exercises

1. **Perplexity by Hand**

   Given a 3-token sequence, the model's output logit probabilities (after softmax) are [0.5, 0.3, 0.2], [0.1, 0.7, 0.2], [0.4, 0.1, 0.5]. Calculate perplexity by hand.

   <details><summary>Hint</summary>Take the log probability for each token, average them, then exponentiate. Lower perplexity means the model is more "confident."</details>

2. **LLM-as-Judge Bias Detection**

   Design an experiment: swap the positions of the same pair of answers (swap answer_a and answer_b), repeat 3 times, and calculate the difference in GPT-4's scores. Compute the Position Bias rate in code.

   <details><summary>Hint</summary>Position Bias rate = number of inconsistent scores before/after swapping / total count. Inconsistency means the judge was influenced by position.</details>

3. **Evaluation Report Writing**

   Using the simulated data below, draw a radar chart, calculate the geometric mean score for both models across 4 benchmarks, and write your conclusions.

   ```python
   scores = {
       'Model A': {'gsm8k': 82, 'mmlu': 71, 'humaneval': 65, 'ifeval': 78},
       'Model B': {'gsm8k': 78, 'mmlu': 75, 'humaneval': 70, 'ifeval': 72},
   }
   ```

   <details><summary>Hint</summary>Geometric mean = (a * b * c * d) ** (1/4). Compare both models' geometric means while also observing which model is more "balanced" on the radar chart.</details>